# Generate Hidden-State Projections and SVD Energy

This Generate notebook computes and caches the data consumed by `Visualize-hidden-state-projections-and-SVD-energy.ipynb`. The companion Visualize notebook presents last-layer hidden-state projections and a cumulative hidden-state SVD-energy summary for the main numerical input families at $N = 500$ using Llama-3.2-3B, as presented under **Attention-Induced Token Graphs: Construction and Visualization Robustness** in the appendix.

Each relative-path archive stores cumulative uncentered-SVD energy for every analyzed layer and the leading three last-layer coordinates used as RGB token-node colors. The notebook requests hidden states only, so attention-head aggregation—including mass versus uniform weighting—does not enter this computation.


Import dependencies; define the model, 500-token analysis window, relative cache schema, and random seeds.


In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import gc
import random
import re
from pathlib import Path

import numpy as np
import torch
from scipy.integrate import solve_ivp
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model, 500-token analysis window, and relative cache path
MODEL_NAME = 'meta-llama/Llama-3.2-3B'
REFERENCE_CONTEXT_TOKENS = 1000
ANALYSIS_CONTEXT_TOKENS = 500
CONTEXT_FRAC = ANALYSIS_CONTEXT_TOKENS / REFERENCE_CONTEXT_TOKENS
CONTEXT_DISPLAY_LABEL = f'{ANALYSIS_CONTEXT_TOKENS} tokens'

PROCESSED_DATA_DIR = Path('processed_hidden_state_data')
CHECKPOINT_EVERY_N_LAYERS = 1
SVD_ENERGY_CHECKPOINT_SCHEMA_VERSION = 'layerwise_hidden_svd_energy_and_final_projection_checkpoint_v2'
HIDDEN_STATE_CONVENTION = 'pre_layer_input_hidden_states_index_layer_minus_1'
DEFAULT_NUM_LAYERS = 28  # Used only to validate an existing cache before loading the model
FINAL_LAYER_PROJECTION_COMPONENTS = 3
DROP_BOS = True

# Seed sequence generation and model inference
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# Input-family order, display labels, and saved-array schema
DATA_ORDER = [
    'Constant 500',
    'Logistic r=3.2',
    'Logistic r=3.5',
    'Logistic r=3.55',
    'Logistic r=3.566',
    'Logistic r=3.5692',
    'Logistic r=3.5698',
    'Logistic r=3.9',
    'Logistic r=4.0',
    'Lorenz',
]

DATA_TITLE_SUFFIXES = {
    'Logistic r=3.2': 'period 2',
    'Logistic r=3.5': 'period 4',
    'Logistic r=3.55': 'period 8',
    'Logistic r=3.566': 'period 16',
    'Logistic r=3.5692': 'period 32',
    'Logistic r=3.5698': 'period 64',
    'Logistic r=3.9': 'chaotic',
    'Logistic r=4.0': 'chaotic',
}

HIDDEN_STATE_ARRAY_KEYS = (
    'layer_indices',
    'svd_cumulative_energy_ratio',
    'final_layer_index',
    'svd_projection_coords',
    'is_delimiter',
)


def panel_title(data_name):
    suffix = DATA_TITLE_SUFFIXES.get(str(data_name))
    return f'{data_name} ({suffix})' if suffix else str(data_name)


def context_display_label(num_tokens=ANALYSIS_CONTEXT_TOKENS):
    return f'{int(num_tokens)} tokens'


def context_fraction_label(frac):
    # Preserve the context50 cache-label convention used by the companion notebook
    pct = int(round(100 * float(frac)))
    return f'context{pct}'


def safe_data_stem(data_name, frac):
    name = str(data_name).replace('=', '_').replace('.', '_')
    name = re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_').lower()
    return f'{name}_{context_fraction_label(frac)}'


def hidden_state_data_path(data_name, frac=CONTEXT_FRAC, data_dir=PROCESSED_DATA_DIR):
    return Path(data_dir) / f'{safe_data_stem(data_name, frac)}_hidden_state_data.npz'


def resolve_all_layer_indices(num_layers):
    num_layers = int(num_layers)
    if num_layers <= 0:
        raise ValueError(f'Invalid num_layers={num_layers}.')
    return np.arange(1, num_layers + 1, dtype=np.int32)


Define representative constant, periodic and chaotic logistic-map, and Lorenz sequences.


In [ ]:
def lorenz(t, y, sigma, rho, beta):
    return [
        sigma * (y[1] - y[0]),
        y[0] * (rho - y[2]) - y[1],
        y[0] * y[1] - beta * y[2],
    ]


# Sample and quantize a representative Lorenz x-coordinate trajectory
sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0
initial_conditions = [-8, 7, 27]
t_span = (0, 50)
t_eval = np.linspace(t_span[0], t_span[1], 500)

sol = solve_ivp(
    lorenz,
    t_span,
    initial_conditions,
    args=(sigma, rho, beta),
    t_eval=t_eval,
)

x_vals = sol.y[0]
x_min = x_vals.min()
x_max = x_vals.max()
x_integers = np.round(50 + (x_vals - x_min) / (x_max - x_min) * 900).astype(int)
N = len(x_integers)
assert N == 500


In [ ]:
# Periodic and chaotic logistic-map regimes
periodic_logistic_periods = {
    3.2: 2,
    3.5: 4,
    3.55: 8,
    3.566: 16,
    3.5692: 32,
    3.5698: 64,
}
logistic_rs = list(periodic_logistic_periods.keys()) + [3.9, 4.0]
logistic_expected_periods = {r: periodic_logistic_periods.get(r) for r in logistic_rs}
x0 = 0.2
logistic_long_n = 20000
logistic_settle_tol = 1e-10


def logistic_sequence(r, x0, n):
    x = np.empty(n, dtype=float)
    x[0] = x0
    for i in range(1, n):
        x[i] = r * x[i - 1] * (1 - x[i - 1])
    return x


def settle_time_float(x, p, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if p is None or p <= 0 or p >= x.size:
        return None
    err = np.abs(x[p:] - x[:-p])
    suffix_max = np.maximum.accumulate(err[::-1])[::-1]
    hits = np.flatnonzero(suffix_max < tol)
    return int(hits[0]) if hits.size else None


def quantize_logistic_window(vals):
    vals = np.asarray(vals, dtype=float)
    vmin, vmax = vals.min(), vals.max()
    if vmax == vmin:
        return np.full(vals.shape[0], 500, dtype=int)
    return np.round(50 + (vals - vmin) / (vmax - vmin) * 900).astype(int)


def build_burned_in_logistic_window(r, x0, n_window, expected_period=None, n_long=20000, tol=1e-10):
    if expected_period is None:
        return logistic_sequence(r, x0, n_window), 0

    x_long = logistic_sequence(r, x0, n_long)
    t_settle = settle_time_float(x_long, expected_period, tol=tol)
    if t_settle is None:
        raise ValueError(f'Could not find settle time for r={r} with expected period {expected_period}.')
    if t_settle + n_window > x_long.size:
        raise ValueError(f'Need a longer trajectory for r={r}; settle index {t_settle} leaves fewer than {n_window} states.')
    return np.asarray(x_long[t_settle:t_settle + n_window], dtype=float), int(t_settle)


# Use settled periodic windows and untrimmed chaotic windows
logistic_sequences = {}
logistic_integers = {}
logistic_window_start_indices = {}
for r in logistic_rs:
    vals, start_idx = build_burned_in_logistic_window(
        r,
        x0=x0,
        n_window=N,
        expected_period=logistic_expected_periods[r],
        n_long=logistic_long_n,
        tol=logistic_settle_tol,
    )
    logistic_sequences[r] = vals
    logistic_integers[r] = quantize_logistic_window(vals)
    logistic_window_start_indices[r] = int(start_idx)

# Assemble one representative 500-state source sequence per input family
input_data_types = {
    'Constant 500': np.full(N, 500, dtype=int),
    'Logistic r=3.2': logistic_integers[3.2],
    'Logistic r=3.5': logistic_integers[3.5],
    'Logistic r=3.55': logistic_integers[3.55],
    'Logistic r=3.566': logistic_integers[3.566],
    'Logistic r=3.5692': logistic_integers[3.5692],
    'Logistic r=3.5698': logistic_integers[3.5698],
    'Logistic r=3.9': logistic_integers[3.9],
    'Logistic r=4.0': logistic_integers[4.0],
    'Lorenz': x_integers,
}

assert list(input_data_types.keys()) == DATA_ORDER


Initialize the tokenizer and lazily load Llama-3.2-3B for hidden-state extraction.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model_hs = None
num_layers = None
hidden_size = None

# Check that the comma delimiter occupies exactly one token
comma_token_ids = tokenizer(',', add_special_tokens=False).input_ids
if len(comma_token_ids) != 1:
    raise ValueError(f"Expected one token for ',', got {comma_token_ids}")
comma_token_id = int(comma_token_ids[0])


def load_hidden_state_model(model_name=MODEL_NAME):
    global model_hs, num_layers, hidden_size
    if model_hs is None:
        # Eager attention matches the companion model setup; this notebook requests no attention matrices,
        # so mass versus uniform attention-head weighting does not apply
        model_hs = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map='auto',
            torch_dtype=torch.float16,
            attn_implementation='eager',
        )
        model_hs.eval()
        num_layers = int(model_hs.config.num_hidden_layers)
        hidden_size = int(model_hs.config.hidden_size)
    return model_hs


Build comma-delimited 500-token prompts, align post-BOS hidden states, and compute uncentered SVD summaries.

Layer convention: one-based layer `L` uses `hidden_states[L - 1]`, the hidden signal entering that transformer block.


In [ ]:
def get_token_start_index(input_ids, bos_token_id=None, drop_bos=True):
    if not drop_bos:
        return 0
    if input_ids is None or bos_token_id is None:
        return 0

    if isinstance(input_ids, torch.Tensor):
        first_id = int(input_ids.flatten()[0].item())
    else:
        first_id = int(np.asarray(input_ids).flatten()[0])

    return 1 if first_id == int(bos_token_id) else 0


def extract_hidden_signal_matrix(hidden_states, input_ids, bos_token_id=None, layer_index=-2, drop_bos=True):
    token_start = get_token_start_index(input_ids, bos_token_id=bos_token_id, drop_bos=drop_bos)

    resolved_layer_index = layer_index if layer_index >= 0 else len(hidden_states) + layer_index
    if resolved_layer_index < 0 or resolved_layer_index >= len(hidden_states):
        raise ValueError(f'Invalid layer_index={layer_index} for {len(hidden_states)} hidden-state tensors')

    signal = hidden_states[resolved_layer_index][0].float()
    # Remove the optional leading BOS position before token-level SVD analysis
    signal = signal[token_start:, :]
    return signal.cpu().numpy(), int(signal.shape[0]), int(resolved_layer_index)


def svd_cumulative_energy_uncentered(signal_matrix):
    X = np.asarray(signal_matrix, dtype=np.float32)
    if X.ndim != 2:
        raise ValueError('signal_matrix must be 2D (num_tokens x hidden_dim)')
    if X.shape[0] == 0:
        return np.zeros((0,), dtype=np.float32)

    # Use squared singular values as the uncentered hidden-state energy spectrum
    singular_values = np.linalg.svd(X, full_matrices=False, compute_uv=False)
    singular_values = np.asarray(singular_values, dtype=np.float32)
    energy = singular_values ** 2
    total = float(np.sum(energy))
    if total <= 0:
        return np.full_like(energy, np.nan, dtype=np.float32)
    return (np.cumsum(energy) / total).astype(np.float32)


def svd_projection_uncentered(signal_matrix, k=FINAL_LAYER_PROJECTION_COMPONENTS):
    X = np.asarray(signal_matrix, dtype=np.float32)
    if X.ndim != 2:
        raise ValueError('signal_matrix must be 2D (num_tokens x hidden_dim)')

    n_tokens = X.shape[0]
    coords = np.zeros((n_tokens, int(k)), dtype=np.float32)
    if n_tokens == 0:
        return {
            'coords': coords,
            'cumulative_energy_ratio': np.zeros((0,), dtype=np.float32),
        }

    # Token coordinates are U times singular values for the leading components
    U, singular_values, _ = np.linalg.svd(X, full_matrices=False)
    singular_values = np.asarray(singular_values, dtype=np.float32)
    n_components = min(int(k), singular_values.shape[0])
    coords[:, :n_components] = U[:, :n_components] * singular_values[:n_components]

    energy = singular_values ** 2
    total = float(np.sum(energy))
    if total <= 0:
        cumulative_energy = np.full_like(energy, np.nan, dtype=np.float32)
    else:
        cumulative_energy = (np.cumsum(energy) / total).astype(np.float32)

    return {
        'coords': coords,
        'cumulative_energy_ratio': cumulative_energy,
    }


def _build_input_sequence_from_frac(sequence, frac, total_n):
    frac_use = float(frac)
    if frac_use >= 1.0:
        ctx_len = int(total_n)
    else:
        ctx_len = max(1, int(round(frac_use * (total_n - 1))))
    ctx_len = min(int(total_n), ctx_len)
    context_window = sequence[:ctx_len]
    # Format each numerical state as three digits followed by a comma
    input_sequence = ','.join([f'{x:03d}' for x in context_window]) + ','
    return input_sequence, ctx_len


Save and validate one archive per input. Separate atomic checkpoints allow interrupted layerwise SVD sweeps to resume safely.


In [ ]:
def _atomic_savez_compressed(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + '.tmp')
    with tmp_path.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    tmp_path.replace(path)


def save_processed_hidden_state_data(data_name, frac, processed_data, data_dir=PROCESSED_DATA_DIR):
    path = hidden_state_data_path(data_name, frac=frac, data_dir=data_dir)
    required_processed_keys = (
        'layer_indices',
        'svd_cumulative_energy_ratio',
        'final_layer_svd_projection_coords',
        'is_delimiter',
    )
    missing = [key for key in required_processed_keys if key not in processed_data]
    if missing:
        raise KeyError(f'Missing required processed arrays: {missing}')

    layer_indices = np.asarray(processed_data['layer_indices'], dtype=np.int32)
    if layer_indices.size == 0:
        raise ValueError('Cannot save hidden-state data without layer indices.')

    arrays_to_save = {
        'layer_indices': layer_indices,
        'svd_cumulative_energy_ratio': np.asarray(processed_data['svd_cumulative_energy_ratio'], dtype=np.float32),
        'final_layer_index': np.asarray(layer_indices[-1], dtype=np.int32),
        'svd_projection_coords': np.asarray(
            processed_data['final_layer_svd_projection_coords'], dtype=np.float32
        ),
        'is_delimiter': np.asarray(processed_data['is_delimiter'], dtype=bool),
    }
    _atomic_savez_compressed(path, **arrays_to_save)
    return path


def load_processed_hidden_state_data(data_name, frac=CONTEXT_FRAC, data_dir=PROCESSED_DATA_DIR):
    path = hidden_state_data_path(data_name, frac=frac, data_dir=data_dir)
    if not path.exists():
        raise FileNotFoundError(path)

    with np.load(path, allow_pickle=False) as npz:
        processed_data = {key: npz[key] for key in npz.files}

    missing = [key for key in HIDDEN_STATE_ARRAY_KEYS if key not in processed_data]
    if missing:
        raise KeyError(f'{path} is missing required arrays: {missing}')
    extra = sorted(set(processed_data) - set(HIDDEN_STATE_ARRAY_KEYS))
    if extra:
        raise KeyError(f'{path} contains unexpected arrays: {extra}')
    return processed_data


def processed_hidden_state_data_complete(data_name, frac=CONTEXT_FRAC, data_dir=PROCESSED_DATA_DIR):
    try:
        processed_data = load_processed_hidden_state_data(data_name, frac=frac, data_dir=data_dir)
    except (FileNotFoundError, OSError, KeyError, ValueError):
        return False

    layer_indices = np.asarray(processed_data['layer_indices'], dtype=np.int32)
    cumulative_energy = np.asarray(processed_data['svd_cumulative_energy_ratio'], dtype=np.float32)
    final_layer_index = int(np.asarray(processed_data['final_layer_index']).item())
    coords = np.asarray(processed_data['svd_projection_coords'], dtype=np.float32)
    is_delimiter = np.asarray(processed_data['is_delimiter'], dtype=bool)

    # Validate shapes against either the loaded model or its documented layer count
    expected_num_layers = int(num_layers) if num_layers is not None else DEFAULT_NUM_LAYERS
    expected_layers = resolve_all_layer_indices(expected_num_layers)
    expected_node_count = int(ANALYSIS_CONTEXT_TOKENS)
    if hidden_size is None:
        expected_modes = cumulative_energy.shape[1] if cumulative_energy.ndim == 2 else -1
    else:
        expected_modes = min(expected_node_count, int(hidden_size))

    return (
        np.array_equal(layer_indices, expected_layers)
        and final_layer_index == expected_num_layers
        and coords.shape == (expected_node_count, FINAL_LAYER_PROJECTION_COMPONENTS)
        and is_delimiter.shape == (expected_node_count,)
        and cumulative_energy.shape == (expected_num_layers, expected_modes)
        and expected_modes > 0
    )


# Store partial layerwise results under the same relative cache directory
def svd_energy_checkpoint_path(data_name, frac=CONTEXT_FRAC, data_dir=PROCESSED_DATA_DIR):
    return Path(data_dir) / '_checkpoints' / f'{safe_data_stem(data_name, frac)}_layerwise_svd_energy.checkpoint.npz'


def initialize_svd_energy_checkpoint(layer_indices, node_count, num_svd_modes, is_delimiter):
    layer_indices = np.asarray(layer_indices, dtype=np.int32)
    n_layers = int(layer_indices.size)
    return {
        'layer_indices': layer_indices,
        'svd_cumulative_energy_ratio': np.full((n_layers, int(num_svd_modes)), np.nan, dtype=np.float32),
        'final_layer_svd_projection_coords': np.full(
            (int(node_count), FINAL_LAYER_PROJECTION_COMPONENTS), np.nan, dtype=np.float32
        ),
        'is_delimiter': np.asarray(is_delimiter, dtype=bool),
        'completed_layers': np.zeros((n_layers,), dtype=bool),
    }


def _npz_scalar_to_str(data, key):
    return str(np.asarray(data[key]).item())


def _checkpoint_metadata_matches(data, data_name, frac, layer_indices, node_count, num_svd_modes, is_delimiter):
    layer_indices = np.asarray(layer_indices, dtype=np.int32)
    n_layers = int(layer_indices.size)
    node_count = int(node_count)
    num_svd_modes = int(num_svd_modes)

    if _npz_scalar_to_str(data, 'checkpoint_schema_version') != SVD_ENERGY_CHECKPOINT_SCHEMA_VERSION:
        return False
    if _npz_scalar_to_str(data, 'model_name') != MODEL_NAME:
        return False
    if _npz_scalar_to_str(data, 'data_name') != str(data_name):
        return False
    if _npz_scalar_to_str(data, 'hidden_state_convention') != HIDDEN_STATE_CONVENTION:
        return False
    if not np.isclose(float(np.asarray(data['context_frac']).item()), float(frac)):
        return False
    if int(np.asarray(data['context_tokens']).item()) != int(ANALYSIS_CONTEXT_TOKENS):
        return False
    if int(np.asarray(data['node_count']).item()) != node_count:
        return False
    if int(np.asarray(data['num_svd_modes']).item()) != num_svd_modes:
        return False
    if bool(np.asarray(data['drop_bos']).item()) != bool(DROP_BOS):
        return False
    if not np.array_equal(data['layer_indices'].astype(np.int32), layer_indices):
        return False
    if not np.array_equal(data['is_delimiter'].astype(bool), np.asarray(is_delimiter, dtype=bool)):
        return False

    expected_shapes = {
        'svd_cumulative_energy_ratio': (n_layers, num_svd_modes),
        'final_layer_svd_projection_coords': (node_count, FINAL_LAYER_PROJECTION_COMPONENTS),
        'completed_layers': (n_layers,),
    }
    return all(data[key].shape == shape for key, shape in expected_shapes.items())


def load_or_initialize_svd_energy_checkpoint(data_name, frac, layer_indices, node_count, num_svd_modes, is_delimiter):
    path = svd_energy_checkpoint_path(data_name, frac=frac)
    if path.exists():
        try:
            with np.load(path, allow_pickle=False) as data:
                if _checkpoint_metadata_matches(
                    data,
                    data_name=data_name,
                    frac=frac,
                    layer_indices=layer_indices,
                    node_count=node_count,
                    num_svd_modes=num_svd_modes,
                    is_delimiter=is_delimiter,
                ):
                    checkpoint = {
                        'layer_indices': data['layer_indices'].astype(np.int32),
                        'svd_cumulative_energy_ratio': data['svd_cumulative_energy_ratio'].astype(np.float32),
                        'final_layer_svd_projection_coords': data[
                            'final_layer_svd_projection_coords'
                        ].astype(np.float32),
                        'is_delimiter': data['is_delimiter'].astype(bool),
                        'completed_layers': data['completed_layers'].astype(bool),
                    }
                    done = int(checkpoint['completed_layers'].sum())
                    print(f'Resuming {data_name}: {done}/{int(checkpoint["layer_indices"].size)} layers already checkpointed.')
                    return checkpoint
                print(f'Ignoring incompatible checkpoint: {path}')
        except (OSError, KeyError, ValueError) as exc:
            print(f'Ignoring unreadable checkpoint {path}: {exc}')

    return initialize_svd_energy_checkpoint(
        layer_indices=layer_indices,
        node_count=node_count,
        num_svd_modes=num_svd_modes,
        is_delimiter=is_delimiter,
    )


def save_svd_energy_checkpoint(data_name, frac, checkpoint, data_dir=PROCESSED_DATA_DIR):
    path = svd_energy_checkpoint_path(data_name, frac=frac, data_dir=data_dir)
    arrays_to_save = {
        'checkpoint_schema_version': np.asarray(SVD_ENERGY_CHECKPOINT_SCHEMA_VERSION),
        'model_name': np.asarray(MODEL_NAME),
        'data_name': np.asarray(str(data_name)),
        'context_frac': np.asarray(float(frac), dtype=np.float64),
        'context_tokens': np.asarray(int(ANALYSIS_CONTEXT_TOKENS), dtype=np.int64),
        'node_count': np.asarray(checkpoint['is_delimiter'].size, dtype=np.int64),
        'num_svd_modes': np.asarray(checkpoint['svd_cumulative_energy_ratio'].shape[1], dtype=np.int64),
        'hidden_state_convention': np.asarray(HIDDEN_STATE_CONVENTION),
        'drop_bos': np.asarray(bool(DROP_BOS)),
        'layer_indices': np.asarray(checkpoint['layer_indices'], dtype=np.int32),
        'svd_cumulative_energy_ratio': np.asarray(checkpoint['svd_cumulative_energy_ratio'], dtype=np.float32),
        'final_layer_svd_projection_coords': np.asarray(
            checkpoint['final_layer_svd_projection_coords'], dtype=np.float32
        ),
        'is_delimiter': np.asarray(checkpoint['is_delimiter'], dtype=bool),
        'completed_layers': np.asarray(checkpoint['completed_layers'], dtype=bool),
    }
    _atomic_savez_compressed(path, **arrays_to_save)
    return path


def processed_data_from_svd_energy_checkpoint(checkpoint):
    return {
        'layer_indices': np.asarray(checkpoint['layer_indices'], dtype=np.int32),
        'svd_cumulative_energy_ratio': np.asarray(checkpoint['svd_cumulative_energy_ratio'], dtype=np.float32),
        'final_layer_svd_projection_coords': np.asarray(
            checkpoint['final_layer_svd_projection_coords'], dtype=np.float32
        ),
        'is_delimiter': np.asarray(checkpoint['is_delimiter'], dtype=bool),
    }


def delete_svd_energy_checkpoint(data_name, frac=CONTEXT_FRAC, data_dir=PROCESSED_DATA_DIR):
    path = svd_energy_checkpoint_path(data_name, frac=frac, data_dir=data_dir)
    if path.exists():
        path.unlink()


Run one hidden-state forward pass per input, compute cumulative SVD energy at every transformer layer, and retain the leading three final-layer coordinates.


In [ ]:
layerwise_svd_energy_internal_cache = {}


def _make_layerwise_svd_energy_cache_key(data_name, frac, ctx_len, layer_indices):
    return (
        str(data_name),
        float(frac),
        int(ctx_len),
        tuple(int(layer_index) for layer_index in layer_indices),
        str(MODEL_NAME),
        bool(DROP_BOS),
        str(HIDDEN_STATE_CONVENTION),
    )


def compute_layerwise_svd_energy_data(data_name, frac=CONTEXT_FRAC):
    if data_name not in input_data_types:
        raise KeyError(f'Unknown data_name={data_name!r}. Available: {list(input_data_types)}')

    sequence = input_data_types[data_name]
    input_sequence, ctx_len = _build_input_sequence_from_frac(sequence, frac, N)

    hs_model = load_hidden_state_model(MODEL_NAME)
    hs_device = next(hs_model.parameters()).device
    num_model_layers = int(hs_model.config.num_hidden_layers)
    hidden_size_use = int(hs_model.config.hidden_size)
    layer_indices = resolve_all_layer_indices(num_model_layers)
    layer_indices_zero_based = layer_indices - 1

    cache_key = _make_layerwise_svd_energy_cache_key(data_name, frac, ctx_len, layer_indices)
    if cache_key in layerwise_svd_energy_internal_cache:
        return layerwise_svd_energy_internal_cache[cache_key]

    # Tokenize the comma-delimited prompt and align saved arrays after optional BOS removal
    inputs = tokenizer(input_sequence, return_tensors='pt').to(hs_device)
    input_ids = inputs.input_ids[0]
    input_ids_np = input_ids.detach().cpu().numpy()
    token_start = get_token_start_index(
        input_ids_np,
        bos_token_id=tokenizer.bos_token_id,
        drop_bos=DROP_BOS,
    )
    token_ids = np.asarray(input_ids_np[token_start:], dtype=np.int64)
    is_delimiter = token_ids == comma_token_id
    node_count = int(token_ids.shape[0])
    if node_count != int(ANALYSIS_CONTEXT_TOKENS):
        raise ValueError(
            f'Expected {ANALYSIS_CONTEXT_TOKENS} non-BOS tokens for {data_name}, '
            f'but tokenizer produced {node_count}.'
        )

    num_svd_modes = min(node_count, hidden_size_use)
    # Reuse only a checkpoint whose model, input, layer, and token metadata all match
    checkpoint = load_or_initialize_svd_energy_checkpoint(
        data_name=data_name,
        frac=frac,
        layer_indices=layer_indices,
        node_count=node_count,
        num_svd_modes=num_svd_modes,
        is_delimiter=is_delimiter,
    )

    if np.all(checkpoint['completed_layers']):
        processed_data = processed_data_from_svd_energy_checkpoint(checkpoint)
        layerwise_svd_energy_internal_cache[cache_key] = processed_data
        del inputs, input_ids
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return processed_data

    # Request hidden states without constructing or returning attention matrices
    with torch.no_grad():
        outputs = hs_model(
            **inputs,
            output_hidden_states=True,
            output_attentions=False,
            return_dict=True,
        )

    if outputs.hidden_states is None:
        raise ValueError('Model did not return hidden states.')
    if len(outputs.hidden_states) < num_model_layers:
        raise ValueError(
            f'Model config reports {num_model_layers} layers, but returned only '
            f'{len(outputs.hidden_states)} hidden-state tensors.'
        )

    completed_since_checkpoint = 0
    checkpoint_every = max(1, int(CHECKPOINT_EVERY_N_LAYERS))
    layer_iter = tqdm(
        list(enumerate(layer_indices_zero_based)),
        desc=f'{data_name} layerwise SVD energy',
        unit='layer',
        leave=False,
        dynamic_ncols=True,
    )
    # Fill any unfinished layer rows and checkpoint at the configured interval
    for saved_layer_offset, layer_idx in layer_iter:
        if checkpoint['completed_layers'][saved_layer_offset]:
            continue

        signal_matrix, seq_len, used_layer = extract_hidden_signal_matrix(
            outputs.hidden_states,
            input_ids=input_ids,
            bos_token_id=tokenizer.bos_token_id,
            layer_index=int(layer_idx),
            drop_bos=DROP_BOS,
        )
        if int(used_layer) != int(layer_idx):
            raise ValueError(f'Hidden-state mismatch: requested {layer_idx}, got hidden-state {used_layer}.')
        if int(seq_len) != node_count:
            raise ValueError(
                f'Node-count mismatch on layer {int(layer_idx) + 1}: '
                f'signal={seq_len}, tokens={node_count}.'
            )

        # Reuse the final-layer SVD to save both its energy curve and 3D coordinates
        is_final_layer = int(layer_idx) == int(layer_indices_zero_based[-1])
        if is_final_layer:
            final_layer_svd = svd_projection_uncentered(
                signal_matrix, k=FINAL_LAYER_PROJECTION_COMPONENTS
            )
            cumulative_energy = final_layer_svd['cumulative_energy_ratio']
            checkpoint['final_layer_svd_projection_coords'] = final_layer_svd['coords']
        else:
            cumulative_energy = svd_cumulative_energy_uncentered(signal_matrix)
        if cumulative_energy.shape[0] != num_svd_modes:
            raise ValueError(f'SVD mode-count mismatch: got {cumulative_energy.shape[0]}, expected {num_svd_modes}.')

        checkpoint['svd_cumulative_energy_ratio'][saved_layer_offset] = cumulative_energy
        checkpoint['completed_layers'][saved_layer_offset] = True
        completed_since_checkpoint += 1

        if completed_since_checkpoint >= checkpoint_every:
            save_svd_energy_checkpoint(data_name, frac, checkpoint)
            completed_since_checkpoint = 0

    if completed_since_checkpoint > 0 or not svd_energy_checkpoint_path(data_name, frac).exists():
        save_svd_energy_checkpoint(data_name, frac, checkpoint)

    if not np.all(checkpoint['completed_layers']):
        missing_layers = checkpoint['layer_indices'][~checkpoint['completed_layers']]
        raise RuntimeError(f'Incomplete checkpoint after generation; missing one-based layers: {missing_layers.tolist()}')

    processed_data = processed_data_from_svd_energy_checkpoint(checkpoint)
    layerwise_svd_energy_internal_cache[cache_key] = processed_data

    # Release large model outputs before advancing to the next input family
    del outputs, inputs, input_ids
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return processed_data


Generate archives in input-family order.


In [ ]:
generated_paths = []
skipped_paths = []

# Reuse compatible archives; rebuild stale or incomplete files
for data_name in tqdm(DATA_ORDER, desc='Hidden-state visualization data', unit='input'):
    output_path = hidden_state_data_path(data_name, frac=CONTEXT_FRAC)
    if processed_hidden_state_data_complete(data_name, frac=CONTEXT_FRAC):
        skipped_paths.append(output_path)
        delete_svd_energy_checkpoint(data_name, frac=CONTEXT_FRAC)
        continue

    if output_path.exists():
        print(f'Rebuilding incomplete or outdated archive: {output_path}')

    processed = compute_layerwise_svd_energy_data(data_name, frac=CONTEXT_FRAC)
    saved_path = save_processed_hidden_state_data(data_name, CONTEXT_FRAC, processed)
    delete_svd_energy_checkpoint(data_name, frac=CONTEXT_FRAC)
    generated_paths.append(saved_path)

print(f'Generated {len(generated_paths)} archive(s); skipped {len(skipped_paths)} existing archive(s).')
print(f'Processed hidden-state data directory: {PROCESSED_DATA_DIR}')
